In [2]:
pip install dash plotly pandas

Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/133.0 kB ? eta -:--:--
     ----------------------------------- -- 122.9/133.0 kB 2.4 MB/s eta 0:00:01
     -------------------------------------- 133.0/133.0 kB 2.6 MB/s eta 0:00:00
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -- ------------------------------------- 0.6/8.9 MB 17.5 MB/s eta 0:00:01
   ------ --------------------------------- 1.5/8.9 MB 18.9 MB/s eta 0:00:01
   ---------- ----------------------------- 2.3/8.9 MB 18.5 MB/s eta 0:00:01
   -------------- ------------------------- 3.2/8.9 MB 18.6 MB/s eta 0:00:01
   ------------------ --------------------- 4.1/8.9 MB 18.6 MB/s eta 0:00:01
   --------------------- ------------------ 4.9/8.9 MB 18.4 MB/s eta 0:00:01
   ------------------------- -------------- 5.6/8.9 MB 17.1 MB/s eta 0:00:01
   ---------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
qcs-api-client 0.21.6 requires attrs<22.0.0,>=21.3.0, but you have attrs 25.4.0 which is incompatible.
qcs-api-client 0.21.6 requires httpx<0.24.0,>=0.23.0, but you have httpx 0.28.1 which is incompatible.
qcs-api-client 0.21.6 requires pydantic<2.0.0,>=1.7.2, but you have pydantic 2.14.1 which is incompatible.


In [3]:
import pandas as pd
import plotly.express as px

from dash import Dash, html, dcc, Input, Output

# Load SpaceX launch data
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
)

spacex_df = pd.read_csv(DATA_URL)

# Find the payload range
min_payload = int(spacex_df["Payload Mass (kg)"].min())
max_payload = int(spacex_df["Payload Mass (kg)"].max())

# Initialize the application
app = Dash(__name__)

# Define the dashboard layout
app.layout = html.Div([
    html.H1(
        "SpaceX Launch Records Dashboard",
        style={"textAlign": "center"}
    ),

    html.Label("Select Launch Site"),

    dcc.Dropdown(
        id="site-dropdown",
        options=[
            {"label": "All Sites", "value": "ALL"}
        ] + [
            {"label": site, "value": site}
            for site in sorted(spacex_df["Launch Site"].unique())
        ],
        value="ALL",
        clearable=False
    ),

    html.Br(),

    dcc.Graph(id="success-pie-chart"),

    html.Label("Payload Mass Range (kg)"),

    dcc.RangeSlider(
        id="payload-slider",
        min=min_payload,
        max=max_payload,
        step=100,
        value=[min_payload, max_payload],
        marks={
            min_payload: str(min_payload),
            max_payload: str(max_payload)
        }
    ),

    html.Br(),

    dcc.Graph(id="success-payload-scatter-chart")
])


# Update the pie chart
@app.callback(
    Output("success-pie-chart", "figure"),
    Input("site-dropdown", "value")
)
def update_pie_chart(selected_site):

    if selected_site == "ALL":
        success_counts = (
            spacex_df.groupby("Launch Site")["class"]
            .sum()
            .reset_index()
        )

        fig = px.pie(
            success_counts,
            values="class",
            names="Launch Site",
            title="Successful Landings by Launch Site"
        )

    else:
        selected_df = spacex_df[
            spacex_df["Launch Site"] == selected_site
        ]

        outcome_counts = (
            selected_df["class"]
            .value_counts()
            .reset_index()
        )

        outcome_counts.columns = ["Outcome", "Count"]
        outcome_counts["Outcome"] = outcome_counts["Outcome"].map({
            1: "Success",
            0: "Failure"
        })

        fig = px.pie(
            outcome_counts,
            values="Count",
            names="Outcome",
            title=f"Landing Outcomes at {selected_site}"
        )

    return fig


# Update the scatter plot
@app.callback(
    Output("success-payload-scatter-chart", "figure"),
    Input("site-dropdown", "value"),
    Input("payload-slider", "value")
)
def update_scatter_chart(selected_site, payload_range):

    filtered_df = spacex_df[
        spacex_df["Payload Mass (kg)"].between(
            payload_range[0],
            payload_range[1]
        )
    ]

    if selected_site != "ALL":
        filtered_df = filtered_df[
            filtered_df["Launch Site"] == selected_site
        ]

    fig = px.scatter(
        filtered_df,
        x="Payload Mass (kg)",
        y="class",
        color="Booster Version Category",
        title="Payload Mass vs. Landing Outcome",
        labels={"class": "Landing Success (0 = No, 1 = Yes)"}
    )

    return fig


if __name__ == "__main__":
    app.run(debug=True)

In [4]:
app.run(jupyter_mode="external", debug=False)

Dash app running on http://127.0.0.1:8050/
